# 第20章：接入 LangGraph 工作流

本篇依据[本章课件](https://github.com/xbsheng/atguigu-note/blob/bbe79814defe76ae597d07dd4f53f2e1fb26ed5d/agent_skills/%E8%AF%BE%E4%BB%B6/%E5%AE%9E%E6%88%98%E7%AF%87/06-%E6%8E%A5%E5%85%A5LangGraph%E5%B7%A5%E4%BD%9C%E6%B5%81.md)整理，对应[视频 P31](https://www.bilibili.com/video/BV14rxM6qEnL?p=31)。资料快照：`bbe79814defe`（2026-10-29）。

以下命令、提示词和示例为课程阅读内容；样本数字是预期核对值，未在本次笔记整理中运行课程服务、安装技能或执行审批。

[完整配套工程](https://github.com/xbsheng/atguigu-note/tree/bbe79814defe76ae597d07dd4f53f2e1fb26ed5d/agent_skills/%E4%BB%A3%E7%A0%81/ecommerce-agent)；[业务口径](https://github.com/xbsheng/atguigu-note/blob/bbe79814defe76ae597d07dd4f53f2e1fb26ed5d/agent_skills/%E8%AF%BE%E4%BB%B6/%E5%AE%9E%E6%88%98%E7%AF%87/%E8%B5%84%E6%96%99/%E4%B8%9A%E5%8A%A1%E9%9C%80%E6%B1%82%E4%B8%8E%E6%95%B0%E6%8D%AE%E5%8F%A3%E5%BE%84.md)；[验收场景](https://github.com/xbsheng/atguigu-note/blob/bbe79814defe76ae597d07dd4f53f2e1fb26ed5d/agent_skills/%E8%AF%BE%E4%BB%B6/%E5%AE%9E%E6%88%98%E7%AF%87/%E8%B5%84%E6%96%99/%E9%AA%8C%E6%94%B6%E5%9C%BA%E6%99%AF%E4%B8%8E%E9%A2%84%E6%9C%9F%E7%BB%93%E6%9E%9C.md)。

## 20.1 方法说明与持久化执行位置

从排查到建工单涉及查数、报告、等待、重查和写入。Skill指导模型怎么做，LangGraph保存状态和下一节点；聊天历史不能代替服务端检查点。

| 内容 | 保存处 |
| --- | --- |
| 查询步骤和报告格式 | Skill |
| 工作流登记与人工决定 | 业务库workflows |
| report、节点状态和执行位置 | LangGraph检查点 |
| 实际工单 | 业务库tickets |

关闭或更换客户端后，凭真实workflow_id和权限仍可查询原流程。

## 20.2 State、Node和Edge

State包含workflow_id、店铺、查询时间、status、report和tickets；节点返回字段更新，边决定下一步。两个业务节点为investigate和review：前者查超时并逐单取证；空报告直接no_action，非空进入review等待。

```mermaid
flowchart TD
    S[START] --> I[investigate：查询与证据分析]
    I --> Q{report 是否为空}
    Q -->|为空| N[no_action：结束]
    Q -->|非空| R[review：interrupt 等待审批]
    H[审批人通过独立入口提交决定] --> D[(workflows.decision)]
    D --> M[MCP resume 读取已有决定]
    M --> R
    R -->|reject| X[rejected：不建工单]
    R -->|approve| W[重查订单并创建或复用工单]
    W --> F[completed]
```

节点使用确定性Python业务逻辑，不再额外调用大模型；Codex已经负责理解自然语言。

### 用两节点图观察完整生命周期

investigate读取当前符合条件订单并逐单取证，有next_offset说明超过单次100单上限，明确失败而不截断。无report直接到END并标no_action；有report进入review，在创建工单前interrupt。

review被恢复时读取业务表中既有decision，拒绝直接rejected，批准则重查已审批每单资格，在事务里复用或创建工单，最后completed。业务判断是Python函数，大模型不在图节点里再次决定是否批准。

## 20.3 interrupt与恢复语义

review在写工单之前interrupt，返回工作流编号及待审批订单，检查点保存为awaiting_approval。恢复用Command(resume=True)，True只表示让图继续，批准与否仍读业务库decision，恢复API先检查决定。

暂停节点恢复时从节点开头重跑，interrupt之前不要放扣库存、邮件或建单副作用。后续写入也要幂等，因为数据库提交后、检查点更新前可能崩溃。

## 20.4 三个编号的不同作用

| 编号 | 来源及用途 |
| --- | --- |
| request_id | 调用者指定，启动重试幂等键 |
| workflow_id | 服务端UUID，查状态、审批和恢复 |
| thread_id | 本工程取相同workflow_id，关联图检查点 |

MCP连接会话id不参与这些映射。原request_id+同参数返回同一工作流；换日期/基准要新编号，复用旧号配新参数返回409。

### 幂等键为什么不能每次重试都换

启动请求发出后超时，服务端可能已经登记并进入等待审批。用原request_id与同参数再调用，start能查到同一个workflow_id并返回状态；换新编号会创建另一流程，掩盖第一次是否执行的不确定性。

相同request_id配不同日期或基准返回409，防止一个业务标识被悄悄改成另一范围。MCP重连只换传输会话，查询业务继续传原workflow_id。

## 20.5 启动与暂停验证

让Agent以固定日期、基准和 `order-demo-001` 启动，展示真实workflow_id和三单report。初始预期awaiting_approval、tickets=[]、decision=null。未审批直接恢复应返回409且无新工单，证明审批在服务端实施。

## 20.6 独立人工决定

审批人在自己的工程终端执行，WORKFLOW_ID替换为真实返回值：

```powershell
uv run --frozen python -m app.approve WORKFLOW_ID --decision approve
```

不同意则选reject分支。approve.py隐藏读取令牌，先获取并显示服务端报告，再输入工作流编号确认，才调用decision接口。相同决定重复提交复用，已提交后不允许改为相反意见。

本地两个令牌仅演示角色：同账号可读.env不构成强隔离。课件要求企业把数据库与审批凭据留在Agent不可读取的服务环境，由独立身份审批。这里仅记录教程，未替用户执行业务审批。

## 20.7 恢复、重查与幂等

Agent先status确认approve/reject，再resume。批准后只遍历已审批report，按原条件重查订单资格：不再满足为skipped_changed；已有工单为existing；首次写入为created。拒绝返回rejected，不建本次工单。

初始三单不变且无已有工单时，新建TK-A1001、TK-A1002、TK-A1003。同流程再恢复返回已完成结果；新流程处理同订单也复用。tickets.order_id唯一是本例范围；允许关闭后再开时要重新定义唯一业务范围。

### 重查资格与重审证据并不相同

批准后的review按原paid_on和as_of重新查询订单是否仍是paid且符合筛选，状态变为shipped/cancelled则跳过。它只处理已审批report，不把调查后新出现的订单添加进清单。

库存、履约若在审批期间变化，本例没有证据版本重审；因此completed只表示按当前实现完成处理。若业务要求新证据使批准失效，需要额外设计版本对比和重审批，不能在报告中假装已有。

## 20.8 重启后恢复所需数据

SqliteSaver由CHECKPOINT_PATH指向持久文件，与MySQL业务库用途不同。等待审批时停止API，再保留相同库和路径启动：

```powershell
uv run --frozen python -m uvicorn app.api:app --host 127.0.0.1 --port 8000 --workers 1
```

查询原workflow_id应仍有报告。只留业务库或只留检查点都不完整。课件提供真实API子进程的验证入口：

```powershell
uv run --frozen pytest -q tests/test_live_stack.py
```

该测试验证协议/状态恢复，不证明模型理解了自然语言，实际是否运行另行记录。

## 20.9 课程实现的规模边界

单API进程用锁串行，单流程最多100单，超限明确失败而非截取前100。需要更大规模时服务端增加受控分批、后台执行、数据库级并发和合适检查点存储，不能直接改workers。

业务库和检查点无统一事务，事务+工单唯一约束减少重复写入。审批依据是调查时报告，恢复只重查订单资格，不重新审批新库存/履约证据；如需证据变更重审、审批过期或审计，要增服务端分支。

## 20.10 实现阅读顺序

State与构图 → investigate → review → start/decide/resume。RLock允许同线程嵌套加锁；数据库事务提交本批工单，唯一约束防重复。status读图状态与已保存决定，检查任务错误并标failed。下面保留完整原程序，便于对照控制流与持久化位置。

### 状态查询如何连接业务库与检查点

record先确认workflow_id属于当前店铺；status读取图快照values，再附上workflows中的decision和decided_at，有节点错误时标failed。start保证请求登记与参数一致，再在没有图状态时启动；decide仅允许awaiting_approval保存一次明确决定；resume先检查已有决定再以同thread_id继续。

RLock只保护单进程并发，工单唯一约束才是数据库级防重的一部分。业务库提交与检查点保存没有统一事务，所以恢复路径要能处理工单已存在的情况。

## 20.11 workflow.py完整代码

interrupt只暂停图，审批权来自数据库记录与API身份，二者不能混用。

[配套文件原文](https://github.com/xbsheng/atguigu-note/blob/bbe79814defe76ae597d07dd4f53f2e1fb26ed5d/agent_skills/%E4%BB%A3%E7%A0%81/ecommerce-agent/app/workflow.py)。代码按资料快照保留。

```python
"""LangGraph 状态管理；单 API 进程内以锁串行处理课程工作流。"""
from datetime import datetime, timezone
from threading import RLock
from typing import TypedDict
from uuid import uuid4
from langgraph.graph import StateGraph, START, END
from langgraph.types import interrupt, Command
from sqlalchemy import select, update
from app import business
from app.db import workflows, tickets, orders


class Conflict(ValueError):
    pass


class State(TypedDict, total=False):
    workflow_id: str
    store: str
    paid_on: str
    as_of: str
    status: str
    report: list[dict]
    tickets: list[dict]


class WorkflowService:
    def __init__(self, engine, checkpointer):
        self.engine = engine
        self.lock = RLock()
        builder = StateGraph(State)
        builder.add_node("investigate", self.investigate)
        builder.add_node("review", self.review)
        builder.add_edge(START, "investigate")
        builder.add_conditional_edges("investigate", lambda state: "review" if state["report"] else END)
        builder.add_edge("review", END)
        self.graph = builder.compile(checkpointer=checkpointer)

    @staticmethod
    def config(workflow_id):
        return {"configurable": {"thread_id": workflow_id}}

    def record(self, workflow_id, store):
        with self.engine.connect() as conn:
            row = conn.execute(select(workflows).where(workflows.c.id == workflow_id,
                                workflows.c.store == store)).mappings().first()
        if row is None:
            raise LookupError("工作流不存在或无权访问")
        return dict(row)

    def investigate(self, state: State):
        page = business.list_orders(self.engine, state["store"], state["paid_on"], state["as_of"], 0, 100)
        if page["next_offset"] is not None:
            raise Conflict("单次工作流最多处理 100 单，请缩小服务端查询范围后再启动；本次未创建工单")
        report = [business.evidence(self.engine, state["store"], row["id"]) for row in page["items"]]
        return {"report": report, "tickets": [], "status": "awaiting_approval" if report else "no_action"}

    def review(self, state: State):
        interrupt({"workflow_id": state["workflow_id"], "order_ids": [r["order"]["id"] for r in state["report"]],
                   "message": "请由审批人通过独立入口核对清单并决定；MCP 恢复工具不具备审批权限"})
        record = self.record(state["workflow_id"], state["store"])
        if record["decision"] not in ("approve", "reject"):
            raise Conflict("尚无人工审批记录")
        if record["decision"] == "reject":
            return {"status": "rejected", "tickets": []}
        results = []
        with self.engine.begin() as conn:
            for item in state["report"]:
                order_id = item["order"]["id"]
                current = conn.execute(select(orders).where(orders.c.id == order_id,
                    business.predicate(state["store"], state["paid_on"], state["as_of"]))
                    .with_for_update()).mappings().first()
                if not current:
                    results.append({"order_id": order_id, "outcome": "skipped_changed"})
                    continue
                existing = conn.execute(select(tickets).where(tickets.c.order_id == order_id)).mappings().first()
                if existing:
                    results.append({**dict(existing), "outcome": "existing"})
                    continue
                ticket = {"id": f"TK-{order_id}", "order_id": order_id,
                          "workflow_id": state["workflow_id"], "status": "open"}
                conn.execute(tickets.insert().values(**ticket))
                results.append({**ticket, "outcome": "created"})
        return {"status": "completed", "tickets": results}

    def status(self, workflow_id, store):
        with self.lock:
            record = self.record(workflow_id, store)
            snapshot = self.graph.get_state(self.config(workflow_id))
            result = dict(snapshot.values)
            if not result:
                result = {"workflow_id": workflow_id, "status": "not_started", "report": [], "tickets": []}
            if any(task.error for task in snapshot.tasks):
                result["status"] = "failed"
            result["decision"] = record["decision"]
            result["decided_at"] = (record["decided_at"].replace(tzinfo=timezone.utc).isoformat()
                                    if record["decided_at"] else None)
            return result

    def start(self, store, paid_on, as_of, request_id):
        business.scope(paid_on, as_of)
        params = {"paid_on": paid_on, "as_of": as_of}
        with self.lock:
            with self.engine.begin() as conn:
                old = conn.execute(select(workflows).where(workflows.c.request_id == request_id)).mappings().first()
                if old and (old["store"] != store or old["params"] != params):
                    raise Conflict("request_id 已用于不同参数，请使用新的请求编号")
                workflow_id = old["id"] if old else str(uuid4())
                if not old:
                    conn.execute(workflows.insert().values(id=workflow_id, request_id=request_id, store=store, params=params))
            config = self.config(workflow_id)
            snapshot = self.graph.get_state(config)
            if not snapshot.values:
                self.graph.invoke({"workflow_id": workflow_id, "store": store, **params}, config)
            return self.status(workflow_id, store)

    def decide(self, workflow_id, store, decision):
        with self.lock:
            record = self.record(workflow_id, store)
            if record["decision"]:
                if record["decision"] != decision:
                    raise Conflict("审批记录已提交，不能改写")
                return self.status(workflow_id, store)
            if self.status(workflow_id, store)["status"] != "awaiting_approval":
                raise Conflict("仅等待审批的工作流允许提交决策")
            with self.engine.begin() as conn:
                conn.execute(update(workflows).where(workflows.c.id == workflow_id).values(
                    decision=decision, decided_at=datetime.now(timezone.utc).replace(tzinfo=None)))
            return self.status(workflow_id, store)

    def resume(self, workflow_id, store):
        with self.lock:
            record = self.record(workflow_id, store)
            config = self.config(workflow_id)
            snapshot = self.graph.get_state(config)
            if not snapshot.next:
                return self.status(workflow_id, store)
            if not record["decision"]:
                raise Conflict("尚未获得人工审批；请审批人通过独立审批入口提交决定")
            self.graph.invoke(Command(resume=True), config)
            return self.status(workflow_id, store)
```